In [ ]:
import numpy as np
from emulator.scale_theta import scale_thetas
from emulator.sigma_model_no_log.model_prob import Emulator21cm,run_inference
import torch
CHECKPOINT_DEFAULT = "emulator/sigma_model_no_log/checkpoints/emulator.pt"

In [2]:
def load_emulator():
    model = Emulator21cm(n_params=6, n_redshifts=3)
    model.load_state_dict(torch.load(CHECKPOINT_DEFAULT, map_location="cpu"))
    return model
model   = load_emulator()
checkpoint_dir = "emulator/sigma_model_no_log/checkpoints"
scalers = np.load(f"{checkpoint_dir}/scalers.npz")

In [3]:
CHECKPOINT_DEFAULT = "checkpoints/emulator.pt"
PARAM_NAMES = ['ALPHA_STAR', 'F_STAR10', 'F_ESC10', 'ALPHA_ESC', 'M_TURN', 't_STAR']

PRIOR_BOUNDS = np.array([
    (0.0, 1.0)
    for _ in PARAM_NAMES
])

N_DIM = len(PARAM_NAMES)

def log_prob(theta, model, scalers, y_obs):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    ps2d_pred, xhi_pred, ps2d_sigma_pred = run_inference(model, theta, scalers=scalers)
    print(xhi_pred.numpy())
    ll = log_likelihood_gaussian(
        y_obs,
        ps2d_pred.numpy().flatten(),
        ps2d_sigma_pred.numpy().flatten(),
    )
    return lp + ll

def log_likelihood_gaussian(
    x_obs:  np.ndarray,
    mu:     np.ndarray,
    sigma:  np.ndarray,
) -> float:
    """
    Gaussian log-likelihood:
      ll = -0.5 * sum[ log(2π) + 2*log(σ) + ((x - μ)/σ)² ]
    """
    x_obs = np.asarray(x_obs, dtype=np.float64)
    mu    = np.asarray(mu,    dtype=np.float64)
    sigma = np.asarray(sigma, dtype=np.float64)

    return float(np.sum(
        - 0.5 * np.log(2 * np.pi)
        - np.log(sigma)
        - 0.5 * ((x_obs - mu) / sigma) ** 2
    ))

def log_prior(theta: np.ndarray) -> float:
    lo, hi = PRIOR_BOUNDS[:, 0], PRIOR_BOUNDS[:, 1]
    return 0.0 if np.all(theta >= lo) and np.all(theta <= hi) else -np.inf


In [4]:
checkpoint_dir = "emulator/sigma_model_no_log/checkpoints"
split       = torch.load(f"{checkpoint_dir}/dataset_split.pt")
test_thetas = split["test_thetas"]
test_ps2d   = split["test_ps2d"]
theta_obs     = test_thetas[2].numpy()
y_obs         = test_ps2d[2].numpy().flatten()

In [5]:
from emulator.scale_theta import scale_thetas, unscale_thetas

In [6]:
unscale_thetas(theta_obs)

array([ 0.80616561, -2.42343365,  0.66692185, -0.51645333,  8.39487806,
        0.57577425])

In [7]:
theta_test = np.array([0.91755712, 1.         , 0.18561411, 0.97272242, 0.11515568, 1.        ])


In [8]:
unscale_thetas(theta_test)

array([ 0.87633568,  0.        , -2.25754356,  0.45908363,  8.23031136,
        1.        ])

In [9]:
#theta_obs = np.array([0.8707771, 0.19218878, 0.91673046, 0.32236445, 0.19743903, 0.57577425])
theta_test = np.array([0.91755712, 1.         , 0.18561411, 0.97272242, 0.11515568, 1.        ])
ps2d_cur, xhi_cur, ps2d_sigma_cur = run_inference(model, theta_obs, scalers = scalers)
y_obs = ps2d_cur.flatten()
print(log_prob(theta_obs, model, scalers, y_obs))
print(log_prob(theta_test, model, scalers, y_obs))


[[0.1980639  0.35341412 0.55247283]]
1863.7769972710998
[[0.18625636 0.34134716 0.5410277 ]]
1844.9875654234695


In [10]:
y_obs.shape

torch.Size([300])

In [11]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import os

def plot_emulator_predictions(
    model,
    scalers,
    theta_truth: np.ndarray,
    theta_test:  np.ndarray,
    y_obs:       np.ndarray,
    kpar:        np.ndarray,
    kperp:       np.ndarray,
    z:           int = 0,
    save_path:   str | None = None,
):
    """
    10x10 grid of bins — each bin shows:
      - Gaussian N(mu, sigma) predicted by the emulator for theta_truth (blue)
      - Gaussian N(mu, sigma) predicted by the emulator for theta_test  (red)
      - Vertical line at y_obs value for that bin (black dashed)

    Parameters
    ----------
    model       : emulator model
    scalers     : scalers passed to run_inference
    theta_truth : (N_DIM,) ground-truth parameters
    theta_test  : (N_DIM,) parameters to compare against truth
    y_obs       : (10, 10) or (100,) observed power spectrum (flattened row-major)
    kpar        : (10,) parallel k-bins
    kperp       : (10,) perpendicular k-bins
    z           : redshift index
    xhi_label   : optional x_hi value for the title
    save_path   : if provided, save figure to this path
    """
    # ── Run emulator for both thetas ─────────────────────────────────────────
    mu_truth,  xhi_label, sigma_truth  = run_inference(model, theta_truth, scalers=scalers)
    mu_test,   _, sigma_test   = run_inference(model, theta_test,  scalers=scalers)
    mu_truth    = mu_truth[0, z].numpy()        # (10, 10)  
    sigma_truth = sigma_truth[0, z].numpy()     # (10, 10)
    mu_test     = mu_test[0, z].numpy()         # (10, 10)
    sigma_test  = sigma_test[0, z].numpy()      # (10, 10)
    y_obs_2d = np.asarray(y_obs).reshape(3, 10, 10)[z]   
    xhi_label = xhi_label[0,z].item()
    # ── Figure setup ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(10, 10, figsize=(22, 20))
    title = "Emulator predictions per bin"
    if xhi_label is not None:
        title += rf" — $x_{{hi}} = {xhi_label:.2f}$"
    fig.suptitle(title, fontsize=15, y=1.01)

    # Shared axis labels
    big_ax = fig.add_subplot(111, frameon=False)
    big_ax.tick_params(labelcolor='none', top=False, bottom=False,
                       left=False, right=False)
    big_ax.set_xlabel(r'$k_\perp$', fontsize=14, labelpad=20)
    big_ax.set_ylabel(r'$k_\parallel$', fontsize=14, labelpad=30)

    COLOR_TRUTH = "steelblue"
    COLOR_TEST  = "crimson"
    COLOR_OBS   = "black"

    for i in range(10):           # kpar index
        for j in range(10):       # kperp index
            ax = axes[9 - i, j]

            mu_t,  sig_t  = mu_truth[i, j],  sigma_truth[i, j]
            mu_e,  sig_e  = mu_test[i, j],   sigma_test[i, j]
            y_val         = y_obs_2d[i, j]

            # x range: cover both gaussians and the observation
            x_min = min(mu_t - 4*sig_t, mu_e - 4*sig_e, y_val)
            x_max = max(mu_t + 4*sig_t, mu_e + 4*sig_e, y_val)
            xx    = np.linspace(x_min, x_max, 300)

            # Gaussian PDFs
            pdf_truth = np.exp(-0.5 * ((xx - mu_t) / sig_t) ** 2) / (sig_t * np.sqrt(2 * np.pi))
            pdf_test  = np.exp(-0.5 * ((xx - mu_e) / sig_e) ** 2) / (sig_e * np.sqrt(2 * np.pi))

            ax.plot(xx, pdf_truth, color=COLOR_TRUTH, linewidth=1.2)
            ax.plot(xx, pdf_test,  color=COLOR_TEST,  linewidth=1.2, linestyle="--")
            ax.fill_between(xx, pdf_truth, alpha=0.15, color=COLOR_TRUTH)
            ax.fill_between(xx, pdf_test,  alpha=0.15, color=COLOR_TEST)

            # Vertical line at y_obs
            y_top = max(pdf_truth.max(), pdf_test.max())
            ax.axvline(y_val, color=COLOR_OBS, linewidth=1.0,
                       linestyle=":", ymax=0.85)

            ax.set_xticks([])
            ax.set_yticks([])

    # ── Bin labels ───────────────────────────────────────────────────────────
    for j in range(10):
        axes[9, j].set_xlabel(f"{kperp[j]:.2f}", fontsize=7)
    for i in range(10):
        axes[9 - i, 0].set_ylabel(f"{kpar[i]:.2f}", fontsize=7,
                                   rotation=0, labelpad=15)

    # ── Legend ───────────────────────────────────────────────────────────────
    legend_handles = [
        mlines.Line2D([], [], color=COLOR_TRUTH, linewidth=1.5,
                      label=r"$\theta_\mathrm{truth}$"),
        mlines.Line2D([], [], color=COLOR_TEST, linewidth=1.5, linestyle="--",
                      label=r"$\theta_\mathrm{test}$"),
        mlines.Line2D([], [], color=COLOR_OBS, linewidth=1.2, linestyle=":",
                      label=r"$y_\mathrm{obs}$"),
    ]
    fig.legend(handles=legend_handles, loc="upper right",
               fontsize=11, framealpha=0.9)

    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Saved to {save_path}")

    plt.show()

In [ ]:
kpar = np.loadtxt("PS1_PS2_Data/bins_kpar.txt")
kperp = np.loadtxt("PS1_PS2_Data/bins_kper.txt")
plot_emulator_predictions(
    model       = model,
    scalers     = scalers,
    theta_truth = theta_obs,
    theta_test  = theta_test,
    y_obs       = y_obs,          # shape (10,10) or (100,)
    kpar        = kpar,
    kperp       = kperp,
    z           = 0,
    save_path   = "figures/emulator/comparison.png",
)
